In [6]:
import torch
import os
import sys 
sys.path.append('/zhome/45/0/155089/G-WEB_Fraud_Detection/src/gweb')  # Add the path where model.py is located

print(os.getcwd())


/dtu/blackhole/0e/154958


In [33]:
import torch
from torch_geometric.loader import NeighborLoader
from model import GCN
from data import AMLtoGraph
import torch_geometric.transforms as T
import typer

def test(
    model_path: str = "/zhome/45/0/155089/G-WEB_Fraud_Detection/models/model_lr-2_83e-04_bs-1024_dropout-0_55_epochs-50.onnx",
    batchsize: int = 256,
    hdn_chnls: int = 16,
    atn_heads: int = 4,
    drop_out: float = 0.6,
) -> None:
    torch.manual_seed(42)

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "mps"
        if torch.backends.mps.is_available()
        else "cpu"
    )

    # Load dataset
    dataset = AMLtoGraph("/dtu/blackhole/0e/154958/data")
    data = dataset[0]

    # Model parameters
    hidden_channels = hdn_chnls
    heads = atn_heads
    dropout = drop_out


    model = GCN(
        in_channels=data.num_features,
        hidden_channels=hidden_channels,
        out_channels=1,
        heads=heads,
        dropout=dropout,
    )
    model.load_state_dict(torch.load(model_path))
    model = model.to(device)
    model.eval()

    split = T.RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
    data = split(data)

    test_loader = NeighborLoader(
        data,
        num_neighbors=[30] * 2,
        batch_size=batchsize,
        input_nodes=data.test_mask,
    )

    total_correct = 0
    total_samples = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for test_data in test_loader:
            test_data.to(device)
            pred = model(
                test_data.x, test_data.edge_index, test_data.edge_attr
            )
            pred.to(device)
            ground_truth = test_data.y
            predictions = (pred > 0.5).float()
            all_preds.extend(predictions.flatten().cpu().numpy())
            all_labels.extend(ground_truth.flatten().cpu().numpy())

            total_correct += (
                (predictions == ground_truth.unsqueeze(1)).sum().item()
            )
            total_samples += len(ground_truth)

    all_preds = [int(i) for i in all_preds]
    all_labels = [int(i) for i in all_labels]

    accuracy = total_correct / total_samples if total_samples > 0 else 0
    print(f"Test Accuracy: {accuracy:.4f}")

    class_names = ["Normal", "Fraud"]



In [2]:
import os
import sys 
sys.path.append('/zhome/45/0/155089/G-WEB_Fraud_Detection/src/gweb')  
import onnx
import onnxruntime as ort
import torch
from torch_geometric.loader import NeighborLoader
from model import GCN
from data import AMLtoGraph
import torch_geometric.transforms as T
import typer


def test(
    model_path: str = "/zhome/45/0/155089/G-WEB_Fraud_Detection/models/model_lr-2_83e-04_bs-1024_dropout-0_55_epochs-50.onnx",
    batchsize: int = 32,
    hdn_chnls: int = 16,
    atn_heads: int = 4,
    drop_out: float = 0.6,
) -> None:
    torch.manual_seed(42)

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "mps"
        if torch.backends.mps.is_available()
        else "cpu"
    )

    # Load dataset
    dataset = AMLtoGraph("/dtu/blackhole/0e/154958/data")
    data = dataset[0]

    # Model parameters
    hidden_channels = hdn_chnls
    heads = atn_heads
    dropout = drop_out

    # Load ONNX model
    onnx_model = onnx.load(model_path)
    ort_session = ort.InferenceSession(model_path)
    

    # Convert to torch tensor for processing
    def predict(data):
        ort_inputs = {
            ort_session.get_inputs()[0].name: data.x.cpu().numpy(),
            ort_session.get_inputs()[1].name: data.edge_index.cpu().numpy(),
        }
        print("The model expects input shape: ", ort_session.get_inputs()[0].shape)
        print("The model expects input shape: ", ort_session.get_inputs()[1].shape)

        ort_outs = ort_session.run(None, ort_inputs)
        return torch.tensor(ort_outs[0])

    # Split data
    split = T.RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
    data = split(data)

    test_loader = NeighborLoader(
        data,
        num_neighbors=[30] * 2,
        batch_size=len(data),
        input_nodes=data.test_mask,
    )

    total_correct = 0
    total_samples = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for test_data in test_loader:
            test_data.to(device)
            pred = predict(test_data)
            ground_truth = test_data.y
            predictions = (pred > 0.5).float()
            all_preds.extend(predictions.flatten().cpu().numpy())
            all_labels.extend(ground_truth.flatten().cpu().numpy())

            total_correct += (
                (predictions == ground_truth.unsqueeze(1)).sum().item()
            )
            total_samples += len(ground_truth)

    all_preds = [int(i) for i in all_preds]
    all_labels = [int(i) for i in all_labels]

    accuracy = total_correct / total_samples if total_samples > 0 else 0
    print(f"Test Accuracy: {accuracy:.4f}")


In [10]:
import onnx
import onnxruntime as ort
import torch
from torch_geometric.loader import NeighborLoader
from model import GCN
from data import AMLtoGraph
import torch_geometric.transforms as T
import typer


def test(
    model_path: str = "/zhome/45/0/155089/G-WEB_Fraud_Detection/models/model_lr-2_83e-04_bs-1024_dropout-0_55_epochs-50.onnx",
    batchsize: int = 32,
    hdn_chnls: int = 16,
    atn_heads: int = 4,
    drop_out: float = 0.6,
) -> None:
    torch.manual_seed(42)

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "mps"
        if torch.backends.mps.is_available()
        else "cpu"
    )

    # Load dataset
    dataset = AMLtoGraph("/dtu/blackhole/0e/154958/data")
    data = dataset[0]

    # Model parameters
    hidden_channels = hdn_chnls
    heads = atn_heads
    dropout = drop_out

    # Load ONNX model
    onnx_model = onnx.load(model_path)
    ort_session = ort.InferenceSession(model_path)
    

    # Convert to torch tensor for processing
    def predict(data):
        ort_inputs = {
            ort_session.get_inputs()[0].name: data.x.cpu().numpy(),
            ort_session.get_inputs()[1].name: data.edge_index.cpu().numpy(),
        }
        print(f"Node feature matrix shape: {data.x.shape}")
        print(f"Total number of nodes: {data.num_nodes}")
        print(f"Total number of edges: {data.edge_index.shape[1]}")

        print("The model expects input shape: ", ort_session.get_inputs()[0].shape)
        print("The model expects input shape: ", ort_session.get_inputs()[1].shape)

        ort_outs = ort_session.run(None, ort_inputs)
        return torch.tensor(ort_outs[0])

    # Split data
    split = T.RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
    data = split(data)

    test_loader = NeighborLoader(
        data,
        num_neighbors=[30] * 2,
        batch_size=len(data),
        input_nodes=data.test_mask,
    )

    total_correct = 0
    total_samples = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for test_data in test_loader:
            test_data.to(device)
            pred = predict(test_data)
            ground_truth = test_data.y
            predictions = (pred > 0.5).float()
            all_preds.extend(predictions.flatten().cpu().numpy())
            all_labels.extend(ground_truth.flatten().cpu().numpy())

            total_correct += (
                (predictions == ground_truth.unsqueeze(1)).sum().item()
            )
            total_samples += len(ground_truth)

    all_preds = [int(i) for i in all_preds]
    all_labels = [int(i) for i in all_labels]

    accuracy = total_correct / total_samples if total_samples > 0 else 0
    print(f"Test Accuracy: {accuracy:.4f}")

In [28]:
import onnx
import onnxruntime as ort

def test(
    model_path: str = "/zhome/45/0/155089/G-WEB_Fraud_Detection/models/model_lr-2_83e-04_bs-1024_dropout-0_55_epochs-50.onnx",
    batchsize: int = 32,
    hdn_chnls: int = 16,
    atn_heads: int = 4,
    drop_out: float = 0.6,
) -> None:
    torch.manual_seed(42)

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "mps"
        if torch.backends.mps.is_available()
        else "cpu"
    )

    # Load dataset
    dataset = AMLtoGraph("/dtu/blackhole/0e/154958/data")
    data = dataset[0]

    # Model parameters
    hidden_channels = hdn_chnls
    heads = atn_heads
    dropout = drop_out

    # Load ONNX model
    onnx_model = onnx.load(model_path)
    ort_session = ort.InferenceSession(model_path)

    # Convert to torch tensor for processing
    def predict(data):
        ort_inputs = {
            ort_session.get_inputs()[0].name: data.x.cpu().numpy(),
            ort_session.get_inputs()[1].name: data.edge_index.cpu().numpy(),
        }
        ort_outs = ort_session.run(None, ort_inputs)
        return torch.tensor(ort_outs[0])

    # Split data
    split = T.RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
    data = split(data)

    test_loader = NeighborLoader(
        data,
        num_neighbors=[30] * 2,
        batch_size=batchsize,
        input_nodes=data.test_mask,
    )

    total_correct = 0
    total_samples = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for test_data in test_loader:
            test_data.to(device)
            pred = predict(test_data)
            ground_truth = test_data.y
            predictions = (pred > 0.5).float()
            all_preds.extend(predictions.flatten().cpu().numpy())
            all_labels.extend(ground_truth.flatten().cpu().numpy())

            total_correct += (
                (predictions == ground_truth.unsqueeze(1)).sum().item()
            )
            total_samples += len(ground_truth)

    all_preds = [int(i) for i in all_preds]
    all_labels = [int(i) for i in all_labels]

    accuracy = total_correct / total_samples if total_samples > 0 else 0
    print(f"Test Accuracy: {accuracy:.4f}")


In [13]:
test()

Node feature matrix shape: torch.Size([2077023, 31])
Total number of nodes: 2077023
Total number of edges: 31898238
The model expects input shape:  [3945, 31]
The model expects input shape:  [2, 27092]


InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Got invalid dimensions for input: x.1 for the following indices
 index: 0 Got: 52 Expected: 3945
 Please fix either the inputs/outputs or the model.

In [6]:
# Load ONNX model
model_path = "/zhome/45/0/155089/G-WEB_Fraud_Detection/models/model_lr-2_83e-04_bs-1024_dropout-0_55_epochs-50.onnx"
onnx_model = onnx.load(model_path)
ort_session = ort.InferenceSession(model_path)

In [7]:
# Check the input names and count
print("Inputs to the ONNX model:")
for i, input_info in enumerate(ort_session.get_inputs()):
    print(f"Input {i}: {input_info.name}, Shape: {input_info.shape}")


Inputs to the ONNX model:
Input 0: x.1, Shape: [3945, 31]
Input 1: edge_index.1, Shape: [2, 27092]
